# NB5 — Type-aware Pairwise V1: S1 smoke + S2.5 tiny overfit

Notebook wrapper cho frozen Core-7 V2 scorer data. Core logic nằm trong `src/scorer/`.
Mục tiêu: verify artifacts → S1 smoke → S2 model checks → S2.5 overfit 32 paired families.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
# Khi test PR, đặt FASHION_REPO_REF=feat/scorer-s2-5-tiny-overfit.
REPO_REF = os.environ.get("FASHION_REPO_REF", "main")
REPO_ROOT = Path.cwd() / "opisoverated"

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

commit = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
print("REPO_ROOT:", REPO_ROOT)
print("REPO_REF :", REPO_REF)
print("Git commit:", commit)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path(
    os.environ.get("FASHION_ARTIFACT_ROOT", "/content/drive/MyDrive/ML_Final")
)
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(
    ARTIFACT_ROOT / "fashionclip_item_embeddings.pt"
)
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(
    ARTIFACT_ROOT / "embedding_manifest_v1.json"
)
os.environ["FASHION_CORE7_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2"
)
os.environ["FASHION_SCORER_READY_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2"
)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


In [ ]:
%pip install -q pyyaml

import json
import yaml
from src.data.runtime_paths import load_runtime_paths
from src.data.build_core7_scorer_dataset import sha256_file

CONFIG_PATH = REPO_ROOT / "configs/scorer_type_aware_pairwise_v1.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert config["model"]["name"] == "type_aware_pairwise_v1"
assert config["model"]["embedding_dim"] == 512
assert config["model"]["category_vocab_size"] == 8
assert config["data"]["min_items"] == 3
assert config["data"]["max_items"] == 8

paths = load_runtime_paths(repo_root=REPO_ROOT)
print("Embedding cache :", paths.embedding_cache)
print("Manifest        :", paths.embedding_manifest)
print("Core-7 dir      :", paths.core7_dir)
print("Scorer-ready dir:", paths.scorer_ready_dir)
print("CONFIG: PASS")


In [ ]:
required = [
    paths.embedding_cache,
    paths.embedding_manifest,
    paths.scorer_ready_dir / "dataset_manifest_v2.json",
    paths.scorer_ready_dir / "split_manifest_v2.json",
    paths.scorer_ready_dir / "final_validation_v2.json",
]
for split in ("train", "valid", "test"):
    required += [
        paths.scorer_ready_dir / f"scorer_ready_v2_{split}.jsonl",
        paths.core7_dir / f"core7_item_metadata_v1_{split}.jsonl",
    ]

missing = [path for path in required if not path.is_file()]
assert not missing, f"Missing artifacts: {missing}"

with (REPO_ROOT / "artifacts/data_v2_reference.json").open("r", encoding="utf-8") as f:
    reference = json.load(f)
freeze = reference["freeze_fields_after_rebuild"]

assert reference["status"] == "READY_TO_TRAIN"
assert reference["dataset_version"] == "polyvore1000-core7-compat-v2"

checks = {
    "embedding_cache": (paths.embedding_cache, freeze["embedding_cache_sha256"]),
    "embedding_manifest": (paths.embedding_manifest, freeze["embedding_manifest_sha256"]),
    "category_mapping": (
        REPO_ROOT / "configs/category_mapping_core7_v2.json",
        freeze["mapping_sha256"],
    ),
}
for split in ("train", "valid", "test"):
    checks[split] = (
        paths.scorer_ready_dir / f"scorer_ready_v2_{split}.jsonl",
        freeze[f"{split}_sha256"],
    )

for name, (path, expected) in checks.items():
    actual = sha256_file(path)
    print(name, "PASS" if actual == expected else "FAIL")
    assert actual == expected, f"{name} hash mismatch"

print("FROZEN V2 HASHES: PASS")


In [ ]:
test_files = [
    "test_scorer_dataset.py",
    "test_pair_generation.py",
    "test_scorer_metrics.py",
    "test_scorer_model.py",
    "test_scorer_training.py",
    "test_scorer_checkpoint.py",
]
for pattern in test_files:
    subprocess.run(
        [
            sys.executable, "-m", "unittest", "discover",
            "-s", "tests", "-p", pattern, "-v",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
print("S1/S2/TRAINING/CHECKPOINT UNIT TESTS: PASS")


In [ ]:
import torch
from functools import partial
from torch.utils.data import DataLoader, Subset
from src.scorer.dataset import (
    EmbeddingStore,
    build_dataset_from_runtime,
    collate_scorer_batch,
    flatten_family_indices,
)

embedding_store = EmbeddingStore(paths.embedding_cache)
train_dataset = build_dataset_from_runtime(
    paths,
    "train",
    embedding_store=embedding_store,
)

assert len(train_dataset) == freeze["train_sample_count"]
assert len(train_dataset.pair_families) * 2 == len(train_dataset)

SMOKE_FAMILIES = 128
families = train_dataset.pair_families[:SMOKE_FAMILIES]
smoke_indices = flatten_family_indices(families)
smoke_dataset = Subset(train_dataset, smoke_indices)

collate_fn = partial(
    collate_scorer_batch,
    max_items=config["data"]["max_items"],
)
smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)
batch = next(iter(smoke_loader))

B = len(batch["sample_ids"])
assert batch["item_embeddings"].shape == (B, 8, 512)
assert batch["coarse_category_ids"].shape == (B, 8)
assert batch["item_mask"].shape == (B, 8)
assert batch["pair_mask"].shape == (B, 8, 8)
assert batch["labels"].shape == (B,)
assert torch.isfinite(batch["item_embeddings"]).all()

item_counts = batch["item_mask"].sum(dim=1)
assert item_counts.min() >= 3 and item_counts.max() <= 8
assert torch.all(batch["item_embeddings"][~batch["item_mask"]] == 0)
assert torch.all(batch["coarse_category_ids"][~batch["item_mask"]] == 0)

actual_pairs = batch["pair_mask"].sum(dim=(1, 2))
expected_pairs = item_counts * (item_counts - 1) // 2
assert torch.equal(actual_pairs, expected_pairs)
assert actual_pairs.max() <= 28

print("Train samples :", len(train_dataset))
print("Pair families :", len(train_dataset.pair_families))
print("Smoke samples :", len(smoke_dataset))
print("BATCH + MASK: PASS")


In [ ]:
pos_idx, neg_idx = families[0]
pos_record = train_dataset.records[pos_idx]
neg_record = train_dataset.records[neg_idx]

assert pos_record["label"] == 1
assert neg_record["label"] == 0
assert neg_record["paired_positive_sample_id"] == pos_record["sample_id"]

differences = [
    i for i, (p, n) in enumerate(zip(pos_record["items"], neg_record["items"]))
    if p != n
]
assert len(differences) == 1

swap_index = differences[0]
original_item = pos_record["items"][swap_index]
replacement_item = neg_record["items"][swap_index]
original_meta = train_dataset.metadata_by_item[original_item]
replacement_meta = train_dataset.metadata_by_item[replacement_item]
assert original_meta["master_category"] == replacement_meta["master_category"]

pos_sample = train_dataset[pos_idx]
first_item_id = pos_sample["item_ids"][0]
cache_row = train_dataset.embedding_row_by_item[first_item_id]
assert torch.allclose(
    pos_sample["item_embeddings"][0],
    train_dataset.embedding_matrix[cache_row].float(),
)
print("LOOKUPS + PAIRING: PASS")


In [ ]:
from src.scorer.evaluate import evaluate_predictions

smoke_records = [train_dataset.records[index] for index in smoke_indices]
labels = [row["label"] for row in smoke_records]
metrics = evaluate_predictions(
    sample_ids=[row["sample_id"] for row in smoke_records],
    paired_positive_sample_ids=[
        row["paired_positive_sample_id"] for row in smoke_records
    ],
    labels=labels,
    logits=[1.0 if label == 1 else -1.0 for label in labels],
)

assert metrics["roc_auc"] == 1.0
assert metrics["fitb_2way"] == 1.0
assert metrics["mean_logit_margin"] == 2.0
assert metrics["median_logit_margin"] == 2.0
assert metrics["sample_count"] == 256
assert metrics["paired_family_count"] == 128

s1_report = {
    "frozen_artifacts": "PASS",
    "unit_tests": "PASS",
    "embedding_lookup": "PASS",
    "category_lookup": "PASS",
    "padding": "PASS",
    "item_mask": "PASS",
    "pair_mask": "PASS",
    "positive_negative_pairing": "PASS",
    "evaluator": "PASS",
    "smoke_sample_count": len(smoke_dataset),
}
print(metrics)
print(s1_report)
print("=" * 72)
print("S1 SCORER SMOKE TEST: PASS")
print("=" * 72)


## S2.5 — Tiny-set overfit

Sanity run trên đúng 32 complete positive-negative families (64 samples). Đây không phải benchmark và không dùng test split.


In [ ]:
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import (
    build_tiny_overfit_loader,
    resolve_device,
    run_tiny_overfit,
    set_reproducible_seed,
)

TINY_FAMILIES = 32
TINY_MAX_EPOCHS = 300
TINY_TARGET_ROC_AUC = 0.99
TINY_TARGET_FITB = 0.99
TINY_MAX_LOSS_RATIO = 0.25
SEED = int(config["training"]["seed"])

set_reproducible_seed(SEED)
tiny_loader, tiny_selection = build_tiny_overfit_loader(
    train_dataset,
    family_count=TINY_FAMILIES,
    batch_size=2 * TINY_FAMILIES,
    seed=SEED,
    max_items=config["data"]["max_items"],
)
model = TypeAwarePairwiseScorer.from_config(config)
device = resolve_device()
print("S2.5 device:", device)
print("S2.5 selection:", tiny_selection)

tiny_result = run_tiny_overfit(
    model,
    tiny_loader,
    config,
    device=device,
    max_epochs=TINY_MAX_EPOCHS,
    expected_family_count=TINY_FAMILIES,
    target_roc_auc=TINY_TARGET_ROC_AUC,
    target_fitb=TINY_TARGET_FITB,
    max_loss_ratio=TINY_MAX_LOSS_RATIO,
    use_amp=False,
)
print("Initial:", tiny_result["initial"])
print("Final  :", tiny_result["final"])
print("Epochs :", tiny_result["epochs_ran"])


In [ ]:
S2_5_RUN_DIR = (
    ARTIFACT_ROOT / "scorer_runs" / "type_aware_pairwise_v1" / commit[:12]
)
S2_5_RUN_DIR.mkdir(parents=True, exist_ok=True)
S2_5_REPORT_PATH = S2_5_RUN_DIR / "s2_5_tiny_overfit_report.json"

s2_5_report = {
    "scorer_version": "type_aware_pairwise_v1",
    "dataset_version": reference["dataset_version"],
    "category_mapping_version": reference["category_mapping_version"],
    "negative_protocol_version": reference["negative_protocol_version"],
    "embedding_version": reference["embedding_version"],
    "git_commit": commit,
    "frozen_input_sha256": {
        "mapping": freeze["mapping_sha256"],
        "embedding_cache": freeze["embedding_cache_sha256"],
        "embedding_manifest": freeze["embedding_manifest_sha256"],
        "train": freeze["train_sha256"],
    },
    "selection": tiny_selection,
    **tiny_result,
}
with S2_5_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(s2_5_report, f, ensure_ascii=False, indent=2)

print("S2.5 report:", S2_5_REPORT_PATH)
if tiny_result["status"] != "PASS":
    raise RuntimeError(
        "S2.5 FAIL: debug labels/lookups/masks/pairs/logits/BCE/optimizer/gradients; "
        "do not start S3."
    )
print("=" * 72)
print("S2.5 TINY OVERFIT: PASS")
print("=" * 72)
